# EMT 3-phase Generic Two-Terminal I-Type SSN

This notebook compares a reference EMT model built from standard MNA components against an SSN-based variant:

1. **Reference case**: voltage source + series RC branch + RL load using standard EMT components
2. **I-type SSN branch case**: the series RC branch is replaced by a generic two-terminal I-type SSN component

The SSN case is validated against the reference simulation using logged CSV results.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import dpsimpy

emt = dpsimpy.emt
ph3 = emt.ph3

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Logger = dpsimpy.Logger
Domain = dpsimpy.Domain
Solver = dpsimpy.Solver
PhaseType = dpsimpy.PhaseType

## Parameters

In [ ]:
time_step = 1e-4
final_time = 0.2
frequency = 50.0

source_voltage = 1.0 * np.exp(1j * 0.0)

i3 = np.eye(3)

branch_resistance = 0.2 * i3
branch_capacitance = 500e-6 * i3

load_resistance = 1.0 * i3
load_inductance = 0.05 * i3

## Helpers

In [ ]:
def abc(phasor):
    return np.array(
        [
            [phasor],
            [phasor * np.exp(-1j * 2 * np.pi / 3)],
            [phasor * np.exp(+1j * 2 * np.pi / 3)],
        ],
        dtype=np.complex128,
    )


def create_rc_branch_itype_ssn_matrices():
    inv_c = np.linalg.inv(branch_capacitance)

    # I-type SSN:
    #
    # x = uC_abc
    # u = i_abc
    # y = v_abc
    #
    # Series RC branch:
    #   duC/dt = C^{-1} i
    #   v      = uC + R i

    a = np.zeros((3, 3))
    b = inv_c
    c = i3.copy()
    d = branch_resistance.copy()

    return a, b, c, d


def sig(df, name):
    cols = [c for c in df.columns if c == name or c.startswith(name + "_")]
    return df[cols].to_numpy()


def load_log(sim_name):
    path = Path("logs") / sim_name / f"{sim_name}.csv"
    df = pd.read_csv(path, skipinitialspace=True)
    df.columns = df.columns.str.strip()
    return df

## Reference case: standard MNA components

In [ ]:
def run_mna_case():
    sim_name = "EMT_Ph3_MNA_RCBranch_RLLoad"

    n1 = emt.SimNode("n1", PhaseType.ABC)
    n2 = emt.SimNode("n2", PhaseType.ABC)
    n3 = emt.SimNode("n3", PhaseType.ABC)
    n4 = emt.SimNode("n4", PhaseType.ABC)

    vs = ph3.VoltageSource("VS")
    vs.set_parameters(abc(source_voltage), frequency)

    r_branch = ph3.Resistor("RBranch")
    r_branch.set_parameters(branch_resistance)

    c_branch = ph3.Capacitor("CBranch")
    c_branch.set_parameters(branch_capacitance)

    r_load = ph3.Resistor("RLoad")
    r_load.set_parameters(load_resistance)

    l_load = ph3.Inductor("LLoad")
    l_load.set_parameters(load_inductance)

    vs.connect([emt.SimNode.gnd, n1])
    r_branch.connect([n1, n2])
    c_branch.connect([n2, n3])

    r_load.connect([n3, n4])
    l_load.connect([n4, emt.SimNode.gnd])

    system = SystemTopology(
        frequency,
        [n1, n2, n3, n4],
        [vs, r_branch, c_branch, r_load, l_load],
    )

    Logger.set_log_dir("logs/" + sim_name)
    logger = Logger(sim_name)
    logger.log_attribute("v3", n3.attr("v"))
    logger.log_attribute("i_branch", r_branch.attr("i_intf"))
    logger.log_attribute("i_load", r_load.attr("i_intf"))

    sim = Simulation(sim_name)
    sim.set_system(system)
    sim.add_logger(logger)
    sim.set_domain(Domain.EMT)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()

## I-type SSN branch case

In [ ]:
def run_rc_branch_itype_ssn_case():
    sim_name = "EMT_Ph3_ITypeSSN_RCBranch_RLLoad"

    n1 = emt.SimNode("n1", PhaseType.ABC)
    # n2 does not exist because the complete RC branch is replaced by SSN
    n3 = emt.SimNode("n3", PhaseType.ABC)
    n4 = emt.SimNode("n4", PhaseType.ABC)

    a, b, c, d = create_rc_branch_itype_ssn_matrices()

    vs = ph3.VoltageSource("VS")
    vs.set_parameters(abc(source_voltage), frequency)

    rc_branch = ph3.GenericTwoTerminalITypeSSN("RCBranch")
    rc_branch.set_parameters(a, b, c, d)

    r_load = ph3.Resistor("RLoad")
    r_load.set_parameters(load_resistance)

    l_load = ph3.Inductor("LLoad")
    l_load.set_parameters(load_inductance)

    vs.connect([emt.SimNode.gnd, n1])
    rc_branch.connect([n1, n3])

    r_load.connect([n3, n4])
    l_load.connect([n4, emt.SimNode.gnd])

    system = SystemTopology(
        frequency,
        [n1, n3, n4],
        [vs, rc_branch, r_load, l_load],
    )

    Logger.set_log_dir("logs/" + sim_name)
    logger = Logger(sim_name)
    logger.log_attribute("v3", n3.attr("v"))
    logger.log_attribute("i_branch", rc_branch.attr("i_intf"))
    logger.log_attribute("v_branch", rc_branch.attr("v_intf"))
    logger.log_attribute("i_load", r_load.attr("i_intf"))

    sim = Simulation(sim_name)
    sim.set_system(system)
    sim.add_logger(logger)
    sim.set_domain(Domain.EMT)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()

## Run simulations

In [ ]:
run_mna_case()
run_rc_branch_itype_ssn_case()

## Load CSV logs

In [ ]:
ref = load_log("EMT_Ph3_MNA_RCBranch_RLLoad")
itype_ssn = load_log("EMT_Ph3_ITypeSSN_RCBranch_RLLoad")

print("Reference columns:")
print(ref.columns.tolist())

print("\nI-type SSN columns:")
print(itype_ssn.columns.tolist())

## Assertions

In [ ]:
t_ref = ref.iloc[:, 0].to_numpy()
t_ssn = itype_ssn.iloc[:, 0].to_numpy()

np.testing.assert_allclose(t_ref, t_ssn)

atol = 1e-9
rtol = 1e-6

np.testing.assert_allclose(sig(ref, "v3"), sig(itype_ssn, "v3"), atol=atol, rtol=rtol)

np.testing.assert_allclose(
    sig(ref, "i_branch"), sig(itype_ssn, "i_branch"), atol=atol, rtol=rtol
)

np.testing.assert_allclose(
    sig(ref, "i_load"), sig(itype_ssn, "i_load"), atol=atol, rtol=rtol
)

print("All assertions passed.")

## Error summary

In [ ]:
signals = ["v3", "i_branch", "i_load"]

rows = []

for name in signals:
    ref_sig = sig(ref, name)
    ssn_sig = sig(itype_ssn, name)
    err = ssn_sig - ref_sig

    for phase in range(ref_sig.shape[1]):
        max_abs_ref = np.max(np.abs(ref_sig[:, phase]))
        max_abs_err = np.max(np.abs(err[:, phase]))

        rows.append(
            {
                "signal": name,
                "phase": phase,
                "max_abs_error": max_abs_err,
                "rms_error": np.sqrt(np.mean(err[:, phase] ** 2)),
                "max_abs_reference": max_abs_ref,
                "relative_max_error": max_abs_err / max(max_abs_ref, 1e-12),
            }
        )

summary = pd.DataFrame(rows)
summary

## Plot comparison

In [ ]:
phase = 0

plt.figure(figsize=(10, 5))

plt.plot(t_ref, sig(ref, "i_branch")[:, phase], label="ref i_branch")
plt.plot(
    t_ssn,
    sig(itype_ssn, "i_branch")[:, phase],
    "--",
    label="ITypeSSN i_branch",
)

plt.xlabel("time [s]")
plt.ylabel("current")
plt.title("Branch current, phase A")
plt.grid(True)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))

plt.plot(t_ref, sig(ref, "i_load")[:, phase], label="ref i_load")
plt.plot(
    t_ssn,
    sig(itype_ssn, "i_load")[:, phase],
    "--",
    label="ITypeSSN i_load",
)

plt.xlabel("time [s]")
plt.ylabel("current")
plt.title("Load current, phase A")
plt.grid(True)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))

plt.plot(t_ref, sig(ref, "v3")[:, phase], label="ref v3")
plt.plot(t_ssn, sig(itype_ssn, "v3")[:, phase], "--", label="ITypeSSN v3")

plt.xlabel("time [s]")
plt.ylabel("voltage")
plt.title("Node voltage v3, phase A")
plt.grid(True)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Plot errors

In [ ]:
for name in signals:
    ref_sig = sig(ref, name)
    ssn_sig = sig(itype_ssn, name)
    err = ssn_sig - ref_sig

    plt.figure(figsize=(10, 5))

    for phase in range(err.shape[1]):
        plt.plot(t_ref, err[:, phase], label=f"phase {phase}")

    plt.xlabel("time [s]")
    plt.ylabel("SSN - reference")
    plt.title(f"Error: {name}")
    plt.grid(True)
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()